# 04. Predictive Machine Learning Modeling & Explainable AI (SHAP)
## Project: AI-Driven Loan Recovery & Risk Analytics

### Overview
This notebook trains, evaluates, and optimizes classification machine learning models to predict the probability of loan recovery:
1. **Preprocessing Pipeline**: Handling numerical scaling and categorical One-Hot Encoding via `ColumnTransformer`.
2. **Model Training & Comparison**:
   - Logistic Regression (Class Weighted)
   - Random Forest Classifier (Balanced)
   - XGBoost Classifier (Imbalance Adjusted)
   - LightGBM Classifier (Champion Model)
3. **Model Evaluation & Recall Optimization (Step 11)**: Decision threshold scanning for debt recovery cash flow maximization.
4. **Explainable AI (SHAP)**: Global Beeswarm, feature importance, and individual borrower decision waterfall.
5. **Model Serialization**: Saving champion pipeline to `models/recovery_predictor_lightgbm.pkl`.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

df = pd.read_csv("../data/processed/loan_recovery_master.csv")
delinquent_df = df[df["overdue_days"] > 30].copy()
delinquent_df["recovery_target"] = delinquent_df["recovery_status"].apply(lambda s: 1 if s in ["Fully Recovered", "Partially Recovered"] else 0)

print(f"Sample Size: {delinquent_df.shape}")
print(delinquent_df["recovery_target"].value_counts(normalize=True))


### 1. Building Preprocessing Pipeline & Splitting Data (Excluding Leaked Post-Settlement Features)

In [ ]:
numeric_features = [
    "credit_score", "annual_income", "loan_amount", "interest_rate",
    "loan_term_months", "overdue_days", "dti_ratio", "ltv_ratio",
    "is_secured", "contact_attempts", "promise_to_pay_kept",
    "delinquency_severity_score", "existing_debts_count", "age"
]

categorical_features = [
    "loan_type", "region", "city_tier", "employment_status",
    "housing_status", "collateral_type", "origination_channel",
    "primary_channel"
]

X = delinquent_df[numeric_features + categorical_features]
y = delinquent_df["recovery_target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)
print("Pipeline preprocessor configured successfully.")


### 2. Model Training & Benchmark Comparison

In [ ]:
scale_pos = round((y_train == 0).sum() / max(1, (y_train == 1).sum()), 2)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=8, class_weight="balanced", random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, scale_pos_weight=scale_pos, random_state=42, eval_metric="logloss"),
    "LightGBM": lgb.LGBMClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, class_weight="balanced", random_state=42, verbose=-1)
}

results = []
fitted_pipes = {}
plt.figure(figsize=(9, 6))

for name, clf in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    fitted_pipes[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, y_proba):.3f})")

plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.title("Comparative ROC-AUC Benchmark", fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.show()

res_df = pd.DataFrame(results).set_index("Model")
display(res_df)


### 3. Step 11: Recall & Decision Threshold Optimization

In [ ]:
lgb_pipe = fitted_pipes["LightGBM"]
y_proba_lgb = lgb_pipe.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.1, 0.9, 33)
thresh_data = []

for t in thresholds:
    p = (y_proba_lgb >= t).astype(int)
    thresh_data.append({
        "threshold": t,
        "recall": recall_score(y_test, p, zero_division=0),
        "precision": precision_score(y_test, p, zero_division=0),
        "f1": f1_score(y_test, p, zero_division=0)
    })
tdf = pd.DataFrame(thresh_data)

plt.figure(figsize=(10, 5))
plt.plot(tdf["threshold"], tdf["recall"], label="Recall", color="green", lw=2)
plt.plot(tdf["threshold"], tdf["precision"], label="Precision", color="blue", lw=2)
plt.plot(tdf["threshold"], tdf["f1"], label="F1-Score", color="purple", lw=2)
plt.axvline(0.175, color="red", linestyle="--", label="Optimal Threshold (0.175)")
plt.title("Recall & Precision Tradeoff Curve for Debt Recovery", fontweight="bold")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()


### 4. Explainable AI (SHAP): Global Feature Contributions & Individual Waterfall

In [ ]:
# Extract transformed features
prep = lgb_pipe.named_steps["preprocessor"]
clf = lgb_pipe.named_steps["classifier"]

ohe_cols = list(prep.named_transformers_["cat"].get_feature_names_out(categorical_features))
all_features = numeric_features + ohe_cols

X_test_trans = pd.DataFrame(prep.transform(X_test), columns=all_features)

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_trans)
shap_mat = shap_values[1] if isinstance(shap_values, list) else shap_values

# SHAP Beeswarm Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_mat, X_test_trans, max_display=12)
